In [1]:
# rag_hybrid_vs_dense.py  (no caching, no EnsembleRetriever import)
# RAG (LangChain + Chroma) with:
#   1) Hybrid Retrieval: BM25 (sparse) + Chroma (dense) via lightweight RRF combiner
#   2) Dense-only baseline (Chroma) for comparison
#
# Usage:
#   pip install -U langchain langchain-openai langchain-community chromadb pypdf python-dotenv pandas
#   export OPENAI_API_KEY=sk-...   # (Windows: setx OPENAI_API_KEY "sk-...")
#   python rag_hybrid_vs_dense.py
#
# Optional: put PDFs/TXTs under ./data to index; otherwise sample texts are used.

import time
from pathlib import Path
from dotenv import load_dotenv
import pandas as pd
from typing import Sequence, Optional, Any, Dict

# ------- LangChain packages -------
# If you're on newer LangChain, prefer:
#   from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain.chat_models import ChatOpenAI
from langchain.embeddings import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import DirectoryLoader, TextLoader, PyPDFLoader
from langchain_community.retrievers import BM25Retriever
from langchain.text_splitter import RecursiveCharacterTextSplitter
# -- IMPORTANT: do NOT import EnsembleRetriever to avoid pgvector/sqlalchemy side-effects
# from langchain.retrievers.ensemble import EnsembleRetriever

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.retrievers import BaseRetriever
from langchain_core.documents import Document

load_dotenv()

# ---------------------------
# Config
# ---------------------------
PERSIST_DIR     = "./chroma_db1"     # Chroma persistence (remove to be in-memory)
DATA_DIR        = "./data"           # PDFs/TXTs go here
COLLECTION      = "docs1"
EMBED_MODEL     = "text-embedding-3-small"
CHAT_MODEL      = "gpt-4o-mini"

# Retrieval sizes
K_DENSE = 4
K_SPARSE = 8
K_HYBRID = 6  # final fused k

# ---------------------------
# Lightweight RRF Ensemble (no external deps)
# ---------------------------
class RRFEnsembleRetriever(BaseRetriever):
    """Combines multiple retrievers using Reciprocal Rank Fusion (RRF)."""
    retrievers: Sequence[Any]
    weights: Optional[Sequence[float]] = None
    k: int = 6            # number of docs to return
    c: float = 60.0       # RRF constant (typical range 10–60)

    def _key(self, d: Document) -> str:
        # Stable-ish dedup key from source + content hash
        return f"{d.metadata.get('source','?')}::{hash(d.page_content)}"

    def _score_docs(self, docs: Sequence[Document], weight: float) -> Dict[str, float]:
        scores: Dict[str, float] = {}
        for rank, d in enumerate(docs):
            key = self._key(d)
            rrf = 1.0 / (self.c + rank + 1)  # reciprocal rank contribution
            scores[key] = max(scores.get(key, 0.0), weight * rrf)
        return scores

    def _get_relevant_documents(self, query: str, *, run_manager=None) -> Sequence[Document]:
        w = self.weights or [1.0] * len(self.retrievers)
        bag: Dict[str, Document] = {}
        fused: Dict[str, float] = {}

        for retr, wt in zip(self.retrievers, w):
            docs = retr.get_relevant_documents(query)
            for d in docs:
                key = self._key(d)
                if key not in bag:
                    bag[key] = d
            partial = self._score_docs(docs, wt)
            for k2, v in partial.items():
                fused[k2] = fused.get(k2, 0.0) + v

        top_keys = sorted(fused.keys(), key=lambda k: fused[k], reverse=True)[: self.k]
        return [bag[k] for k in top_keys]

    async def _aget_relevant_documents(self, query: str, *, run_manager=None) -> Sequence[Document]:
        return self._get_relevant_documents(query, run_manager=run_manager)

# ---------------------------
# Helpers
# ---------------------------
def load_docs():
    """Load docs from ./data (PDF/TXT). Fall back to small samples."""
    docs = []
    data_path = Path(DATA_DIR)
    if data_path.exists():
        # pass 1: text-like files (DirectoryLoader + TextLoader)
        loader = DirectoryLoader(
            DATA_DIR,
            glob="**/*",
            loader_cls=TextLoader,
            show_progress=True,
            use_multithreading=True,
        )
        try:
            docs.extend(loader.load())
        except Exception:
            pass
        # pass 2: PDFs (PyPDFLoader)
        for pdf in data_path.rglob("*.pdf"):
            try:
                docs.extend(PyPDFLoader(str(pdf)).load())
            except Exception:
                pass

    if not docs:
        from langchain.schema import Document
        docs = [
            Document(page_content=("LangChain is a framework for developing LLM apps. "
                                   "It integrates vector stores like Chroma and supports RAG pipelines."),
                     metadata={"source": "sample:langchain"}),
            Document(page_content=("Chroma is an open-source embedding DB (vector store) that stores "
                                   "document embeddings and enables similarity search."),
                     metadata={"source": "sample:chroma"}),
        ]
    return docs


def build_or_load_chroma(embeddings) -> Chroma:
    """Create/load a persistent Chroma index using plain embeddings (no cache)."""
    Path(PERSIST_DIR).mkdir(parents=True, exist_ok=True)
    index_exists = (Path(PERSIST_DIR) / "chroma.sqlite3").exists()
    if index_exists:
        return Chroma(
            collection_name=COLLECTION,
            embedding_function=embeddings,
            persist_directory=PERSIST_DIR,
        )

    docs = load_docs()
    splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=150)
    chunks = splitter.split_documents(docs)

    vs = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        collection_name=COLLECTION,
        persist_directory=PERSIST_DIR,
    )
    vs.persist()
    return vs


def build_bm25_from_vstore_text(vstore: Chroma) -> BM25Retriever:
    """
    Build a BM25 retriever using the same chunking used for Chroma.
    (We re-split from original docs to keep parity.)
    """
    docs = load_docs()
    splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=150)
    chunks = splitter.split_documents(docs)

    bm25 = BM25Retriever.from_documents(chunks)
    bm25.k = K_SPARSE
    return bm25


def make_prompt():
    return ChatPromptTemplate.from_messages(
        [
            ("system",
             "You are a concise, helpful assistant. Use the provided context to answer.\n"
             "If the answer isn't in the context, say you don't know.\n\n"
             "Cite sources inline at the end as [source:<short>].\n\n"
             "Context:\n{context}"),
            ("human", "{question}"),
        ]
    )


def format_docs(docs):
    return "\n\n".join(
        f"Source: {d.metadata.get('source','?')}\n{d.page_content}" for d in docs
    )


def make_rag_chain(retriever, llm):
    """RAG chain: retrieve → prompt → LLM → string."""
    prompt = make_prompt()
    chain = (
        {
            "context": retriever | (lambda docs: format_docs(docs)),
            "question": RunnablePassthrough(),
        }
        | prompt
        | llm
        | StrOutputParser()
    )
    return chain


def timed(fn):
    def _inner(*args, **kwargs):
        t0 = time.time()
        out = fn(*args, **kwargs)
        return out, time.time() - t0
    return _inner


def sources_short(docs):
    """Compact semicolon-joined sources list for logging."""
    return ";".join([str(d.metadata.get("source","?")) for d in docs])


# ---------------------------
# Main
# ---------------------------
if __name__ == "__main__":
    # 1) Plain embeddings (no cache)
    embeddings = OpenAIEmbeddings(model=EMBED_MODEL)

    # 2) Persistent Chroma index
    vstore = build_or_load_chroma(embeddings)

    # Dense-only retriever (baseline)
    dense_retriever = vstore.as_retriever(search_kwargs={"k": K_DENSE})

    # Sparse retriever (BM25)
    bm25_retriever = build_bm25_from_vstore_text(vstore)

    # 3) Hybrid retriever via lightweight RRF (no Ensemble import)
    hybrid_retriever = RRFEnsembleRetriever(
        retrievers=[bm25_retriever, dense_retriever],
        weights=[0.55, 0.45],
        k=K_HYBRID,
        c=60.0,
    )

    # 4) LLM
    llm = ChatOpenAI(model=CHAT_MODEL, temperature=0)

    # 5) RAG chains
    chain_dense = make_rag_chain(dense_retriever, llm)
    chain_hybrid = make_rag_chain(hybrid_retriever, llm)

    # 6) Evaluation input CSV
    input_csv = r"C:\Users\surya.adatravu\Documents\RAGAnalysis\RA_FSM_QA.csv"
    df = pd.read_csv(input_csv)

    # Output columns
    df["t_dense"] = 0.0
    df["t_hybrid"] = 0.0
    df["ans_dense"] = ""
    df["ans_hybrid"] = ""
    df["src_dense"] = ""
    df["src_hybrid"] = ""

    # Retrieve + answer
    for i in range(df.shape[0]):
        question = str(df.loc[i, "Question"])

        dense_docs = dense_retriever.get_relevant_documents(question)
        hybrid_docs = hybrid_retriever.get_relevant_documents(question)

        (ans_d), t_d = timed(chain_dense.invoke)(question)
        df.loc[i, "t_dense"] = round(t_d, 3)
        df.loc[i, "ans_dense"] = ans_d
        df.loc[i, "src_dense"] = sources_short(dense_docs)

        (ans_h), t_h = timed(chain_hybrid.invoke)(question)
        df.loc[i, "t_hybrid"] = round(t_h, 3)
        df.loc[i, "ans_hybrid"] = ans_h
        df.loc[i, "src_hybrid"] = sources_short(hybrid_docs)

    # 7) Save comparison
    out_csv = "results_hybrid_vs_dense_1.csv"
    df.to_csv(out_csv, index=False)
    print("\nDone. Comparison written to:", Path(out_csv).resolve())
    print(f"• Chroma DB: {Path(PERSIST_DIR).resolve()}")


C:\Users\surya.adatravu\AppData\Local\anaconda3\Lib\site-packages\langchain_core\_api\deprecation.py:119: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 0.2.0. An updated version of the class exists in the langchain-openai package and should be used instead. To use it run `pip install -U langchain-openai` and import as `from langchain_openai import OpenAIEmbeddings`.
  warn_deprecated(
100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 46.06it/s]
C:\Users\surya.adatravu\AppData\Local\anaconda3\Lib\site-packages\langchain_core\_api\deprecation.py:119: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 0.2.0. An updated version of the class exists in the langchain-openai package and should be used instead. To use it run `pip install -U langchain-openai` and import as `from langchain_openai import ChatOpenA


Done. Comparison written to: C:\Users\surya.adatravu\Documents\RAG_HRETREIVER\results_hybrid_vs_dense_1.csv
• Chroma DB: C:\Users\surya.adatravu\Documents\RAG_HRETREIVER\chroma_db1
